In [140]:
"""
Synthetic Hiking Review Dataset Generator

Creates ~100-200 reviews per trail over 4 years using Ollama for
review text generation.

Features:
- Real Pacific Northwest trails
- Seasonal weather patterns
- Elevation-aware snow conditions
- Rare bear encounters
- Trail-specific wildlife
- Trail-specific navigation issues
- Weekend parking pressure
- Realistic ratings
- Elevation gain included in dataset
"""


'\nSynthetic Hiking Review Dataset Generator\n\nCreates ~100-200 reviews per trail over 4 years using Ollama for\nreview text generation.\n\nFeatures:\n- Real Pacific Northwest trails\n- Seasonal weather patterns\n- Elevation-aware snow conditions\n- Rare bear encounters\n- Trail-specific wildlife\n- Trail-specific navigation issues\n- Weekend parking pressure\n- Realistic ratings\n- Elevation gain included in dataset\n'

In [141]:
import random
from datetime import datetime, timedelta
import pandas as pd
import ollama
import time

In [157]:
OLLAMA_MODEL = "gemma3:latest"

START_DATE = datetime(2021, 1, 1)
END_DATE = datetime(2025, 12, 31)

MIN_REVIEWS_PER_TRAIL = 50
MAX_REVIEWS_PER_TRAIL = 100

In [158]:
TRAILS = [
    {
        "name": "Silver Star Mountain via South Ridge",
        "state": "Washington",
        "lat": 45.7820,
        "lon": -122.2920,
        "difficulty": 4,
        "elevation_gain_ft": 4300,
        "peak_elevation_ft": 4390,
        "landmarks": [
            "wildflower meadows",
            "summit views",
            "rocky ridge"
        ],
        "navigation_probability": 0.08,
        "wildlife": ["deer", "hawk", "chipmunk", "rabbit"]
    },

    {
        "name": "McNeil Point",
        "state": "Oregon",
        "lat": 45.3733,
        "lon": -121.7356,
        "difficulty": 4,
        "elevation_gain_ft": 2800,
        "peak_elevation_ft": 7100,
        "landmarks": [
            "McNeil Point shelter",
            "Mt Hood views",
            "alpine meadows"
        ],
        "navigation_probability": 0.14,
        "wildlife": ["deer", "elk", "hawk"]
    },

    {
        "name": "Angel's Rest to Devil's Rest Loop",
        "state": "Oregon",
        "lat": 45.5605,
        "lon": -122.1720,
        "difficulty": 3,
        "elevation_gain_ft": 2200,
        "peak_elevation_ft": 3240,
        "landmarks": [
            "Angel's Rest viewpoint",
            "Devil's Rest",
            "Columbia Gorge views"
        ],
        "navigation_probability": 0.18,
        "wildlife": ["deer", "chipmunk", "hawk"]
    },

    {
        "name": "Nesmith Point Trail",
        "state": "Oregon",
        "lat": 45.6815,
        "lon": -121.9020,
        "difficulty": 4,
        "elevation_gain_ft": 3800,
        "peak_elevation_ft": 3744,
        "landmarks": [
            "long climb",
            "ridge views",
            "forest sections"
        ],
        "navigation_probability": 0.05,
        "wildlife": ["deer", "elk", "rabbit"]
    },

    {
        "name": "Mt Defiance via Starvation Creek Trail",
        "state": "Oregon",
        "lat": 45.6900,
        "lon": -121.7400,
        "difficulty": 5,
        "elevation_gain_ft": 5000,
        "peak_elevation_ft": 4960,
        "landmarks": [
            "Starvation Creek",
            "waterfalls",
            "summit ridge"
        ],
        "navigation_probability": 0.10,
        "wildlife": ["deer", "elk", "hawk"]
    },

    {
        "name": "Larch Mountain",
        "state": "Oregon",
        "lat": 45.5396,
        "lon": -122.0866,
        "difficulty": 3,
        "elevation_gain_ft": 1500,
        "peak_elevation_ft": 4055,
        "landmarks": [
            "Sherrard Point",
            "viewpoint",
            "forest trail"
        ],
        "navigation_probability": 0.03,
        "wildlife": ["deer", "chipmunk"]
    },

    {
        "name": "Mt. St Helens Summit via Ptarmigan Trail",
        "state": "Washington",
        "lat": 46.2010,
        "lon": -122.1880,
        "difficulty": 5,
        "elevation_gain_ft": 4500,
        "peak_elevation_ft": 8363,
        "landmarks": [
            "boulder fields",
            "summit crater",
            "volcanic landscape"
        ],
        "navigation_probability": 0.10,
        "wildlife": ["mountain goat", "ptarmigan"]
    },

    {
        "name": "Dog Mountain",
        "state": "Washington",
        "lat": 45.7087,
        "lon": -121.7113,
        "difficulty": 4,
        "elevation_gain_ft": 2800,
        "peak_elevation_ft": 2890,
        "landmarks": [
            "wildflower slopes",
            "summit views",
            "Columbia Gorge"
        ],
        "navigation_probability": 0.05,
        "wildlife": ["hawk", "deer", "chipmunk"]
    },

    {
        "name": "Hamilton Mountain",
        "state": "Washington",
        "lat": 45.6654,
        "lon": -121.9648,
        "difficulty": 3,
        "elevation_gain_ft": 2100,
        "peak_elevation_ft": 2480,
        "landmarks": [
            "waterfalls",
            "pool of the winds",
            "ridge views"
        ],
        "navigation_probability": 0.05,
        "wildlife": ["deer", "hawk"]
    },

    {
        "name": "Ramona Falls",
        "state": "Oregon",
        "lat": 45.3738,
        "lon": -121.8269,
        "difficulty": 2,
        "elevation_gain_ft": 1100,
        "peak_elevation_ft": 3560,
        "landmarks": [
            "Ramona Falls",
            "river crossing",
            "forest trail"
        ],
        "navigation_probability": 0.12,
        "wildlife": ["deer", "bird", "salamander"]
    }
]

In [159]:
def weighted_choice(options):
    return random.choices(
        population=list(options.keys()),
        weights=list(options.values()),
        k=1
    )[0]

In [160]:
def weather_for_month(month):

    if month in [6, 7, 8]:
        return weighted_choice({
            "sunny": 70,
            "partly cloudy": 15,
            "cloudy": 10,
            "rain": 5
        })

    if month in [9, 10]:
        return weighted_choice({
            "sunny": 40,
            "partly cloudy": 20,
            "cloudy": 20,
            "rain": 15,
            "foggy": 5
        })

    if month in [11, 12, 1, 2]:
        return weighted_choice({
            "cloudy": 40,
            "rain": 35,
            "foggy": 15,
            "sunny": 10
        })

    return weighted_choice({
        "sunny": 30,
        "partly cloudy": 20,
        "cloudy": 25,
        "rain": 20,
        "foggy": 5
    })


In [161]:
def snow_probability(peak_elevation_ft, month):

    if peak_elevation_ft > 7000:
        probs = {
            1: .98, 2: .98, 3: .95, 4: .90,
            5: .75, 6: .40, 7: .10, 8: .02,
            9: .05, 10: .30, 11: .75, 12: .95
        }

    elif peak_elevation_ft > 4500:
        probs = {
            1: .90, 2: .90, 3: .80, 4: .60,
            5: .35, 6: .15, 7: .02, 8: .00,
            9: .01, 10: .10, 11: .40, 12: .75
        }

    else:
        probs = {
            1: .25, 2: .20, 3: .10, 4: .03,
            5: .01, 6: .00, 7: .00, 8: .00,
            9: .00, 10: .01, 11: .05, 12: .15
        }

    return probs[month]

In [162]:
#Add something that says its a less popular trail

def overgrowth_probability(month):

    if month in [5, 6, 7]:
        return 0.18

    if month in [8, 9]:
        return 0.10

    if month in [3, 4]:
        return 0.08

    return 0.02

In [163]:
def bear_probability(month):

    if month in [6, 7, 8]:
        return 0.025

    if month in [4, 5, 9, 10]:
        return 0.015

    return 0.002

In [164]:

def generate_context(trail, review_date):

    month = review_date.month
    weekend = review_date.weekday() >= 5

    weather = weather_for_month(month)

    snow_present = (
        random.random()
        < snow_probability(
            trail["peak_elevation_ft"],
            month
        )
    )

    snowshoes_needed = (
        snow_present
        and random.random() < 0.35
    )

    bear_seen = (
        random.random()
        < bear_probability(month)
    )

    wildlife = None

    if bear_seen:
        wildlife = "bear"

    elif random.random() < 0.35:
        wildlife = random.choice(
            trail["wildlife"]
        )

    overgrowth = (
        random.random()
        < overgrowth_probability(month)
    )

    navigation_issue = (
        random.random()
        < trail["navigation_probability"]
    )

    parking = "plenty"

    crowded_trails = {
        "Dog Mountain",
        "Hamilton Mountain",
        "Angel's Rest to Devil's Rest Loop",
        "Ramona Falls"
    }

    if trail["name"] in crowded_trails and weekend:

        parking = weighted_choice({
            "full by 8am": 35,
            "limited": 35,
            "moderate": 20,
            "plenty": 10
        })

    landmark = random.choice(
        trail["landmarks"]
    )

    return {
        "weather": weather,
        "snow_present": snow_present,
        "snowshoes_needed": snowshoes_needed,
        "wildlife": wildlife,
        "overgrowth": overgrowth,
        "navigation_issue": navigation_issue,
        "parking": parking,
        "landmark": landmark
    }


In [165]:
def generate_rating(trail, context):

    base = 4.4 - (trail["difficulty"] * 0.18)

    if context["snowshoes_needed"]:
        base -= 0.2

    if context["overgrowth"]:
        base -= 0.4

    if context["navigation_issue"]:
        base -= 0.3

    if context["parking"] in [
        "full by 8am",
        "limited"
    ]:
        base -= 0.2

    rating = round(
        min(
            5,
            max(
                1,
                random.gauss(base, 0.8)
            )
        )
    )

    return int(rating)


In [166]:
system_prompt = f"""

    You are an experienced hiker writing realistic trail reviews
    similar to reviews found on AllTrails.
    
    Requirements:
    - Write 1 to 5 sentences.
    - Sound like a real hiker.
    - Usually positive but not overly enthusiastic.
    - Mention trail conditions when relevant.
    - Sometimes use imperfect grammar or punctuation.
    - Do not mention every piece of information provided.
    - Do not invent major events that were not supplied.
    - No bullet points.
    - No headings.
    - No explanations.
    - Output only the review text.

    """

In [167]:
def generate_review_text(
    trail,
    review_date,
    rating,
    context
):
    user_prompt = f"""

    Write a realistic hiking review similar to what would
    appear on AllTrails using the information below:
    
    
    Trail:
    {trail["name"]}
    
    Date:
    {review_date.strftime("%Y-%m-%d")}
    
    Difficulty:
    {trail["difficulty"]}/5
    
    Elevation Gain:
    {trail["elevation_gain_ft"]} ft
    
    Rating:
    {rating}/5
    
    Conditions:
    Weather={context["weather"]}
    Snow={context["snow_present"]}
    Snowshoes={context["snowshoes_needed"]}
    Wildlife={context["wildlife"]}
    Overgrowth={context["overgrowth"]}
    Parking={context["parking"]}
    NavigationIssue={context["navigation_issue"]}
    Landmark={context["landmark"]}
    
    Write only the review.

    """

    
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }]
    )

    return (
        response["message"]["content"]
        .strip()
    )



In [168]:
def random_date():

    total_days = (
        END_DATE - START_DATE
    ).days

    return START_DATE + timedelta(
        days=random.randint(
            0,
            total_days
        )
    )

In [169]:
def generate_dataset():

    rows = []
    review_id = 1

    for trail in TRAILS:

        review_count = random.randint(
            MIN_REVIEWS_PER_TRAIL,
            MAX_REVIEWS_PER_TRAIL
        )

        print(
            f"Generating {review_count} reviews "
            f"for {trail['name']}"
        )
        trail_start = time.perf_counter()


        for _ in range(review_count):

            review_date = random_date()

            context = generate_context(
                trail,
                review_date
            )

            rating = generate_rating(
                trail,
                context
            )

            review_text = generate_review_text(
                trail,
                review_date,
                rating,
                context
            )

            rows.append({
                "review_id": review_id,
                "trail_name": trail["name"],
                "state": trail["state"],
                "latitude": trail["lat"],
                "longitude": trail["lon"],
                "difficulty": trail["difficulty"],
                "elevation_gain_ft": trail["elevation_gain_ft"],
                "rating": rating,
                "date": review_date.date().isoformat(),
                "review_text": review_text
            })

            review_id += 1
            
        trail_elapsed = (
                time.perf_counter() - trail_start
            )
    
        print(
                f"Completed {trail['name']} | "
                f"{trail_elapsed:.1f}s"
            )


    return pd.DataFrame(rows)


In [170]:
if __name__ == "__main__":

    df = generate_dataset()

    df.to_csv(
        "synthetic_hiking_reviews.csv",
        index=False
    )

    print("\nDataset complete.")
    print("Rows:", len(df))
    print(df.head())

Generating 70 reviews for Silver Star Mountain via South Ridge
Completed Silver Star Mountain via South Ridge | 70 reviews | 234.8s
Generating 63 reviews for McNeil Point
Completed McNeil Point | 63 reviews | 195.9s
Generating 84 reviews for Angel's Rest to Devil's Rest Loop
Completed Angel's Rest to Devil's Rest Loop | 84 reviews | 288.5s
Generating 84 reviews for Nesmith Point Trail
Completed Nesmith Point Trail | 84 reviews | 271.5s
Generating 81 reviews for Mt Defiance via Starvation Creek Trail
Completed Mt Defiance via Starvation Creek Trail | 81 reviews | 272.6s
Generating 57 reviews for Larch Mountain
Completed Larch Mountain | 57 reviews | 172.3s
Generating 65 reviews for Mt. St Helens Summit via Ptarmigan Trail
Completed Mt. St Helens Summit via Ptarmigan Trail | 65 reviews | 217.9s
Generating 97 reviews for Dog Mountain
Completed Dog Mountain | 97 reviews | 292.5s
Generating 64 reviews for Hamilton Mountain
Completed Hamilton Mountain | 64 reviews | 193.5s
Generating 83 revi